# 技能0 · Day 6 上机：研究方法论入门 -- 营销 AI 领域文献计量

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **arxiv** 包查询真实 arXiv API，获取营销 AI 领域真实论文元数据
2. 用 **pandas** 做文献计量统计（按年份/作者/主题）
3. 用 **networkx** 构建作者合作网络与关键词共现网络
4. 用 **matplotlib** 可视化论文增长趋势与网络结构
5. 理解可复现研究（OSF 预注册 / FAIR 原则）为什么是 2026 年学术研究的基本要求

## 营销映射
本 Day 把"研究方法论"桥接到 AI + 企业营销：查询 arXiv "marketing analytics" / "causal inference marketing" / "LLM marketing" 主题论文，做文献计量分析，相当于营销技术选型前的"学术尽职调查"。


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> 需要的库：arxiv（arXiv API 客户端）、pandas（文献计量）、networkx（网络分析）、matplotlib（可视化）。
> arXiv API 有速率限制（约 1 请求/3 秒），高频查询会收到 HTTP 429/503，此时自动切换到 fallback 样本。


In [ ]:
# !pip install arxiv pandas networkx matplotlib -q


## 1. 数据集背景与营销映射

**数据源**：arXiv API 实时查询（首选）或 `data/arxiv_fallback_sample.json`（fallback，18 篇真实论文元数据快照）。

| 查询主题 | arXiv query | 营销映射 | 研究方法论意义 |
|---------|------------|---------|-------------|
| marketing analytics | `"marketing analytics"` | 营销分析技术成熟度 | 文献综述第一步：系统检索 |
| causal inference marketing | `"causal inference marketing"` | 营销归因与增量建模基础 | 因果推断是营销归因的理论基础 |
| LLM marketing | `"LLM marketing"` | LLM 在营销中的应用前沿 | 新兴方向识别 |

**文献计量学（Bibliometrics）** 是研究方法论的核心方法之一：用统计方法分析学术文献的量化特征（论文数/引用数/合作网络/关键词共现），发现领域研究热度演化与新兴方向。


In [ ]:
import arxiv
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
import os
from collections import Counter
from itertools import combinations

# Fallback data path
FALLBACK_PATH = os.path.join('data', 'arxiv_fallback_sample.json')

def load_fallback():
    """Load fallback arXiv sample when API is rate-limited (HTTP 429/503)."""
    with open(FALLBACK_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data['papers']

print("Libraries loaded.")
print("arxiv version:", arxiv.__version__)
print("pandas version:", pd.__version__)
print("networkx version:", nx.__version__)
print("Fallback path:", FALLBACK_PATH, "- exists:", os.path.exists(FALLBACK_PATH))


## TODO 1：用 arxiv 包查询 arXiv API（含 fallback）

**任务**：查询三个主题的 arXiv 论文，获取真实元数据。当 API 被速率限制（HTTP 429/503）或网络不通时，自动切换到 `data/arxiv_fallback_sample.json`。

**提示**：
- `arxiv.Client(num_retries=1, page_size=20)` 创建客户端
- `arxiv.Search(query=q, max_results=20, sort_by=arxiv.SortCriterion.Relevance)` 按关键词搜索
- 遍历 `client.results(search)`，提取 `r.entry_id`/`r.title`/`r.authors`/`r.published`/`r.summary`/`r.primary_category`
- `r.entry_id` 形如 `http://arxiv.org/abs/2210.03629v1`，用 `split('/')[-1]` 提取 arXiv ID
- 异常时用 `load_fallback()` 加载 fallback 样本

**理论连接**：文献综述的第一步是系统检索相关文献。arxiv 包让这一步可编程化。fallback 机制体现**可复现研究**原则--即使 API 不可用，分析也能复现。


In [ ]:
# 查询 arXiv API，获取营销 AI 领域真实论文元数据（含 fallback 机制）
QUERIES = ["marketing analytics", "causal inference marketing", "LLM marketing"]
all_papers = []
api_ok = False

try:
    client = arxiv.Client(num_retries=1, page_size=20)
    for q in QUERIES:
        search = arxiv.Search(query=q, max_results=20, sort_by=arxiv.SortCriterion.Relevance)
        for r in client.results(search):
            arxiv_id = r.entry_id.split('/')[-1]
            all_papers.append({
                'arxiv_id': arxiv_id,
                'title': r.title,
                'authors': [str(a) for a in r.authors],
                'published': r.published.strftime('%Y-%m-%d'),
                'year': r.published.year,
                'query': q,
                'primary_category': r.primary_category,
                'summary': r.summary[:200]
            })
    api_ok = True
    print(f"arXiv API OK: fetched {len(all_papers)} papers across {len(QUERIES)} queries")
except Exception as e:
    print(f"arXiv API failed ({type(e).__name__}: {str(e)[:80]}), using fallback sample")
    all_papers = load_fallback()

print(f"Total papers: {len(all_papers)}")
print(f"Data source: {'arXiv API (real-time)' if api_ok else 'fallback sample (real arXiv snapshot)'}")
print(f"Queries: {QUERIES}")


## TODO 2：文献计量 -- 按年份统计论文增长趋势

**任务**：用 pandas 将论文列表转为 DataFrame，按年份统计论文数，观察营销 AI 领域的研究热度演化。

**提示**：
- `pd.DataFrame(all_papers)` 从列表创建 DataFrame
- `df['year'].value_counts().sort_index()` 按年份统计并排序
- 打印各年论文数，识别增长趋势

**数据治理视角**：年份字段应该是整数类型（int），检查是否有缺失值。论文增长趋势是判断一个技术领域"是否成熟"的关键信号--增长曲线越陡，说明学术界关注度越高。


In [ ]:
# 文献计量 -- 按年份统计论文增长趋势
df = pd.DataFrame(all_papers)
year_counts = df['year'].value_counts().sort_index()

print("=== Papers by Year ===")
print(year_counts)
print(f"\nYear range: {year_counts.index.min()} - {year_counts.index.max()}")
print(f"Peak year: {year_counts.idxmax()} ({year_counts.max()} papers)")
print(f"Total papers: {len(df)}")


## TODO 3：高产作者排名与主题分布

**任务**：用 pandas 统计高产作者排名（Top 10）和按查询主题的论文分布。

**提示**：
- `df.groupby('query').size()` 按主题统计论文数
- `df.explode('authors')` 展开作者列表（一篇论文的多个作者各占一行）
- `authors_exploded['authors'].value_counts().head(10)` 统计高产作者 Top 10

**营销映射**：高产作者是领域的"意见领袖"，识别他们有助于追踪前沿研究方向。主题分布揭示三个查询主题的论文密度差异。


In [ ]:
# 高产作者排名与主题分布
topic_counts = df.groupby('query').size().sort_values(ascending=False)
print("=== Papers by Query Topic ===")
print(topic_counts)

authors_exploded = df.explode('authors')
author_counts = authors_exploded['authors'].value_counts().head(10)
print("\n=== Top 10 Prolific Authors ===")
print(author_counts)
print(f"\nTotal unique authors: {authors_exploded['authors'].nunique()}")


## TODO 4：作者合作网络（networkx）

**任务**：用 networkx 构建作者合作网络（节点=作者，边=合作关系），计算度中心性识别核心作者。

**提示**：
- `nx.Graph()` 创建无向图
- 遍历每篇论文的作者列表，用 `itertools.combinations(authors, 2)` 生成所有作者对
- `G.add_edge(a, b, weight=1)` 添加合作边（已存在则 weight+1）
- `nx.degree_centrality(G)` 计算度中心性
- `sorted(deg_cent.items(), key=lambda x: -x[1])[:5]` 取 Top 5

**理论连接**：合作网络是文献计量学的核心方法。度中心性（Degree Centrality）= deg(v)/(n-1)，衡量作者的协作广度。高中心性作者是领域的"枢纽节点"。


In [ ]:
# 作者合作网络（networkx）
G_collab = nx.Graph()

for _, row in df.iterrows():
    authors = row['authors']
    for a, b in combinations(authors, 2):
        if G_collab.has_edge(a, b):
            G_collab[a][b]['weight'] += 1
        else:
            G_collab.add_edge(a, b, weight=1)

print(f"=== Author Collaboration Network ===")
print(f"Nodes (authors): {G_collab.number_of_nodes()}")
print(f"Edges (collaborations): {G_collab.number_of_edges()}")
print(f"Connected components: {nx.number_connected_components(G_collab)}")

deg_cent = nx.degree_centrality(G_collab)
top_authors_net = sorted(deg_cent.items(), key=lambda x: -x[1])[:5]
print("\nTop 5 authors by degree centrality:")
for name, cent in top_authors_net:
    print(f"  {name}: {cent:.4f} (degree={G_collab.degree(name)})")


## TODO 5：关键词共现网络（networkx）

**任务**：用 networkx 构建关键词共现网络（节点=论文标题中的词，边=同一标题中共现），识别新兴研究方向。

**提示**：
- 对每篇论文标题分词（`title.lower().replace(':','').replace(',','').split()`）
- 过滤短词（`len(w) > 3`），用 `set()` 去重
- 用 `combinations(sorted(set(words)), 2)` 生成词对
- `G.add_edge(a, b, weight=1)` 添加共现边
- `sorted(G.degree, key=lambda x: -x[1])[:10]` 取 Top 10 关键词

**营销映射**：关键词共现网络揭示领域"概念地图"。高频共现词对是核心研究方向，低频新出现的词对可能是新兴方向。


In [ ]:
# 关键词共现网络（networkx）
G_kw = nx.Graph()

for _, row in df.iterrows():
    title = row['title'].lower().replace(':', '').replace(',', '').replace('-', ' ')
    words = [w for w in title.split() if len(w) > 3]
    unique_words = sorted(set(words))
    for a, b in combinations(unique_words, 2):
        if G_kw.has_edge(a, b):
            G_kw[a][b]['weight'] += 1
        else:
            G_kw.add_edge(a, b, weight=1)

print(f"=== Keyword Co-occurrence Network ===")
print(f"Nodes (keywords): {G_kw.number_of_nodes()}")
print(f"Edges (co-occurrences): {G_kw.number_of_edges()}")

top_kw = sorted(G_kw.degree, key=lambda x: -x[1])[:10]
print("\nTop 10 keywords by degree:")
for kw, deg in top_kw:
    print(f"  {kw}: degree={deg}")

top_edges = sorted(G_kw.edges(data=True), key=lambda x: -x[2]['weight'])[:5]
print("\nTop 5 co-occurring keyword pairs:")
for a, b, d in top_edges:
    print(f"  {a} -- {b}: weight={d['weight']}")


## TODO 6：可视化论文增长趋势与合作网络（matplotlib）

**任务**：用 matplotlib 绘制两个图：① 论文增长趋势柱状图 ② 作者合作网络图（最大连通子图）。

**提示**：
- `fig, axes = plt.subplots(1, 2, figsize=(14, 5))` 创建 1x2 子图
- `axes[0].bar(year_counts.index.astype(str), year_counts.values)` 柱状图
- `max(nx.connected_components(G), key=len)` 取最大连通子图
- `nx.spring_layout(sub, seed=42)` 力导向布局
- `nx.draw(sub, pos, ax=axes[1], node_size=30, with_labels=False)` 绘制网络
- `plt.savefig('output/day6_bibliometrics.png', dpi=100, bbox_inches='tight')` 保存

**可复现研究**：`seed=42` 固定随机种子，确保任何人复现都能得到相同的网络布局图。


In [ ]:
# 可视化论文增长趋势与合作网络（matplotlib）
os.makedirs('output', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Papers by year (bar chart)
years_str = year_counts.index.astype(str)
axes[0].bar(years_str, year_counts.values, color='steelblue', edgecolor='navy', alpha=0.8)
axes[0].set_title('Marketing AI Papers by Year', fontsize=13)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Papers')
for i, v in enumerate(year_counts.values):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontsize=10)

# Subplot 2: Author collaboration network (largest connected component)
largest_cc = max(nx.connected_components(G_collab), key=len)
sub = G_collab.subgraph(largest_cc)
pos = nx.spring_layout(sub, seed=42, k=0.6)
node_sizes = [G_collab.degree(n) * 20 for n in sub.nodes()]
nx.draw(sub, pos, ax=axes[1], node_size=node_sizes, node_color='coral',
        edge_color='gray', alpha=0.7, with_labels=False, width=0.3)
# Label top-3 authors by degree
top3 = sorted(sub.degree, key=lambda x: -x[1])[:3]
for name, _ in top3:
    axes[1].annotate(name.split()[-1], pos[name], fontsize=7, ha='center')
axes[1].set_title(f'Author Collaboration Network ({len(largest_cc)} nodes)')

plt.tight_layout()
plt.savefig('output/day6_bibliometrics.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to output/day6_bibliometrics.png")


## 2. 反思与前沿

### 反思问题
1. 营销 AI 领域论文增长趋势如何？哪个年份论文最多？
2. 高产作者是谁？他们合作形成了怎样的网络结构？
3. 关键词共现网络揭示了哪些新兴方向？
4. 如果你要在营销领域做一个 A/B 测试研究，如何用 OSF 预注册？

### 2026 前沿：可复现研究 + ASReview + LLM 辅助文献综述

- **可复现研究**：`seed=42` 固定随机种子、`requirements.txt` 锁定依赖版本--确保任何人都能复现你的文献计量分析
- **OSF 预注册**：在数据收集前公开注册研究假设，对抗 p-hacking 与发表偏倚
- **FAIR 原则**：数据与代码应 Findable/Accessible/Interoperable/Reusable
- **ASReview**（`pip install asreview`）：AI 辅助系统性文献综述，用主动学习筛选论文，比人工快 10x
- **LLM 辅助研究**：用 DeepSeek 等开源 LLM 做文献摘要提取，但必须用 arxiv 包验证论文真实存在（防止 LLM 幻觉）

> ⚠️ LLM 辅助文献综述是加速工具，不是替代工具。LLM 可能幻觉出不存在的论文--这就是为什么本 Day 用 arxiv 包查询**真实 API 返回的真实论文元数据**。
